# 01 — EDA: WM-811K Wafer Defect Detection

Goal: understand class distribution, wafer-map dimensions, and per-defect visual patterns before building the preprocessing stage.

**Note the imbalance.** Most wafers are `none`. Document this — it drives the preprocessing strategy.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Load dataset

Download `LSWMD.pkl` from Kaggle and place it in `../data/`.

In [ ]:
df = pd.read_pickle('../data/LSWMD.pkl')
print(df.shape)
df.head()

In [ ]:
# failureType is stored as nested arrays — normalize to a clean string column
df['failureType'] = df['failureType'].apply(
    lambda x: x[0][0] if isinstance(x, np.ndarray) and x.size > 0 else 'unlabeled'
)
df['failureType'].value_counts()

## Class distribution

Separate labeled vs. unlabeled. The ~172k labeled subset is what you train on.

In [ ]:
labeled = df[df['failureType'] != 'unlabeled'].copy()
print('Labeled wafers:', len(labeled))

plt.figure(figsize=(10, 4))
labeled['failureType'].value_counts().plot(kind='bar')
plt.title('Defect class distribution (labeled subset)')
plt.ylabel('count')
plt.yscale('log')
plt.tight_layout()
plt.show()

## Wafer-map dimensions

Maps vary in size — confirm the range so the preprocessing resize target is justified.

In [ ]:
labeled['dim'] = labeled['waferMap'].apply(lambda m: m.shape)
print(labeled['dim'].value_counts().head(10))

## Visualize one example per defect type

In [ ]:
types = [t for t in labeled['failureType'].unique() if t != 'none']
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, t in zip(axes.ravel(), types):
    sample = labeled[labeled['failureType'] == t]['waferMap'].iloc[0]
    ax.imshow(sample)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Notes / decisions

- Resize target: ___
- Imbalance strategy: ___
- Classes to keep / merge: ___